# `dippa.fitting` walkthrough

Same two jobs as the previous notebook: a fast manual check while
developing, and a first example for users once the API stabilises.

This one demonstrates the actual fitter — not just evaluating known
parameters (see `01_profiles_walkthrough.ipynb`), but finding them from a
genuinely bad starting guess. See `AUDIT.md` §11 for why the fitting
algorithm works the way it does (local decontaminated windows, not "freeze
other peaks and fit against the whole pattern").

## 1. Load the real reference pattern

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.io
from pathlib import Path

from dippa.background import fit_background_quadratic
from dippa.fitting import fit_pattern
from dippa.profiles import evaluate_pattern

fixtures = Path("../tests/fixtures")
aa_real = scipy.io.loadmat(fixtures / "reference_fit.mat")["aa"]
data = scipy.io.loadmat(fixtures / "reference_data.mat")["data"]
x, y = data[:, 0], data[:, 1]
n_peaks = aa_real.shape[1] - 1
print(f"{n_peaks} peaks, {len(x)} measured points")

## 2. Build a deliberately bad starting guess

A real user supplies rough peak positions (e.g. by clicking on a plot) —
that's the one thing kept close to right here, just jittered within the
fitting window. Everything else (amplitude, width, mixing parameter,
background) is set to a generic placeholder or a large random perturbation,
not read from the known answer.

In [ ]:
rng = np.random.default_rng(42)
aa_guess = aa_real.copy()

for i in range(n_peaks):
    aa_guess[0, i] += rng.uniform(-0.005, 0.005)  # jitter position
    aa_guess[1, i] *= rng.uniform(0.5, 1.5)         # +/-50% amplitude
    aa_guess[2, i] = 0.002                           # generic width guess
    aa_guess[3, i] = 0.5                             # generic eta guess
    aa_guess[4, i] = 0.002
    aa_guess[5, i] = 0.5

# Background re-derived from the (perturbed) guessed positions, not taken
# from the known answer either.
c0, c1, c2 = fit_background_quadratic(x, y, aa_guess[0, :n_peaks], half_width=0.02)
aa_guess[0, -1], aa_guess[1, -1], aa_guess[2, -1] = c0, c1, c2

def r_squared(y, model):
    ss_res = np.sum((y - model) ** 2)
    ss_tot = np.sum((y - y.mean()) ** 2)
    return 1 - ss_res / ss_tot

model_guess = evaluate_pattern(x, aa_guess)
print(f"Starting R^2: {r_squared(y, model_guess):.3f}  (negative = worse than predicting the mean)")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(x, y, ".", markersize=1, alpha=0.3, label="measured")
ax.plot(x, model_guess, linewidth=1, label="starting guess")
ax.set_xlabel("g (1/d, A^-1)")
ax.set_ylabel("intensity")
ax.set_title("Before fitting: the starting guess doesn't resemble the data at all")
ax.legend()
plt.show()

## 3. Run the staged fitter

In [ ]:
aa_fitted = fit_pattern(x, y, aa_guess, half_width=0.02, n_passes=3)
model_fitted = evaluate_pattern(x, aa_fitted)
print(f"Final R^2: {r_squared(y, model_fitted):.4f}")

position_errors = np.abs(aa_fitted[0, :n_peaks] - aa_real[0, :n_peaks])
print(f"Largest peak-position error vs the real saved fit: {position_errors.max():.6f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(x, y, ".", markersize=1, alpha=0.3, label="measured")
ax.plot(x, model_fitted, linewidth=1, label="fitted (3 passes)")
ax.set_xlabel("g (1/d, A^-1)")
ax.set_ylabel("intensity")
ax.set_title("After fitting: recovered from a genuinely bad starting point")
ax.legend()
plt.show()

## 4. Convergence across passes

The fitter is staged: each pass refits every peak once, seeing the latest
estimate of every other peak from the previous pass (a Gauss-Seidel-style
scheme — see `AUDIT.md` §11 for why this is a deliberate simplification
versus the original tool's joint fitting of overlapping peaks). Worth
seeing how much of the improvement happens in the first pass versus later
ones.

In [ ]:
aa_progress = aa_guess.copy()
r2_by_pass = [r_squared(y, evaluate_pattern(x, aa_progress))]

for _ in range(4):
    aa_progress = fit_pattern(x, y, aa_progress, half_width=0.02, n_passes=1)
    r2_by_pass.append(r_squared(y, evaluate_pattern(x, aa_progress)))

fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(range(len(r2_by_pass)), r2_by_pass, "o-")
ax.set_xlabel("pass number (0 = starting guess)")
ax.set_ylabel("R^2 against real data")
ax.set_title("Convergence across passes")
plt.show()